# Advanced Pandas
## GroupBy, merging, pivot tables, and performance optimization

## 1. Sample Data Setup

In [1]:
import pandas as pd
import numpy as np

# Create sample sales data
sales_data = {
    'Date': pd.date_range('2023-01-01', periods=12, freq='MS'),
    'Department': ['Sales', 'IT', 'HR', 'Sales', 'IT', 'HR'] * 2,
    'Product': ['A', 'B', 'C', 'A', 'B', 'C'] * 2,
    'Amount': [100, 200, 150, 120, 210, 160, 110, 220, 155, 130, 215, 165],
    'Quantity': [10, 20, 15, 12, 21, 16, 11, 22, 15, 13, 21, 16]
}

df_sales = pd.DataFrame(sales_data)
print("Sales Data:")
print(df_sales.head())

Sales Data:
        Date Department Product  Amount  Quantity
0 2023-01-01      Sales       A     100        10
1 2023-02-01         IT       B     200        20
2 2023-03-01         HR       C     150        15
3 2023-04-01      Sales       A     120        12
4 2023-05-01         IT       B     210        21


## 2. GroupBy Operations

In [2]:
# GroupBy with single column
grouped_dept = df_sales.groupby('Department')['Amount'].sum()
print("Total amount by department:")
print(grouped_dept)

# GroupBy with multiple columns
grouped_multi = df_sales.groupby(['Department', 'Product'])['Amount'].sum()
print("\nTotal amount by department and product:")
print(grouped_multi)

Total amount by department:
Department
HR       630
IT       845
Sales    460
Name: Amount, dtype: int64

Total amount by department and product:
Department  Product
HR          C          630
IT          B          845
Sales       A          460
Name: Amount, dtype: int64


In [4]:
# Multiple aggregations
agg_results = df_sales.groupby('Department')['Amount'].agg(['sum', 'mean', 'count', 'std'])
print("Multiple aggregations:")
print(agg_results)

# Named aggregations
agg_named = df_sales.groupby('Department').agg(
    total_amount=('Amount', 'sum'),
    avg_amount=('Amount', 'mean'),
    num_records=('Amount', 'count')
)
print("\nNamed aggregations:")
print(agg_named)

Multiple aggregations:
            sum    mean  count        std
Department                               
HR          630  157.50      4   6.454972
IT          845  211.25      4   8.539126
Sales       460  115.00      4  12.909944

Named aggregations:
            total_amount  avg_amount  num_records
Department                                       
HR                   630      157.50            4
IT                   845      211.25            4
Sales                460      115.00            4


In [5]:
# Transform - broadcast result back to original shape
df_sales['dept_total'] = df_sales.groupby('Department')['Amount'].transform('sum')
print("GroupBy transform:")
print(df_sales[['Department', 'Amount', 'dept_total']].head())

# Calculate percentage of total
df_sales['pct_of_dept'] = df_sales['Amount'] / df_sales['dept_total']
print("\nPercentage of department total:")
print(df_sales[['Department', 'Amount', 'pct_of_dept']].head())

GroupBy transform:
  Department  Amount  dept_total
0      Sales     100         460
1         IT     200         845
2         HR     150         630
3      Sales     120         460
4         IT     210         845

Percentage of department total:
  Department  Amount  pct_of_dept
0      Sales     100     0.217391
1         IT     200     0.236686
2         HR     150     0.238095
3      Sales     120     0.260870
4         IT     210     0.248521


## 3. Merging and Joining

In [6]:
# Create sample datasets
df_employees = pd.DataFrame({
    'EmployeeID': [1, 2, 3, 4],
    'Name': ['Alice', 'Bob', 'Charlie', 'David'],
    'Department': ['Sales', 'IT', 'HR', 'Sales']
})

df_salaries = pd.DataFrame({
    'EmployeeID': [1, 2, 3, 5],
    'Salary': [50000, 60000, 55000, 52000]
})

print("Employees:")
print(df_employees)
print("\nSalaries:")
print(df_salaries)

Employees:
   EmployeeID     Name Department
0           1    Alice      Sales
1           2      Bob         IT
2           3  Charlie         HR
3           4    David      Sales

Salaries:
   EmployeeID  Salary
0           1   50000
1           2   60000
2           3   55000
3           5   52000


In [7]:
# Inner merge
inner_merge = pd.merge(df_employees, df_salaries, on='EmployeeID', how='inner')
print("Inner merge:")
print(inner_merge)

# Left merge
left_merge = pd.merge(df_employees, df_salaries, on='EmployeeID', how='left')
print("\nLeft merge:")
print(left_merge)

# Right merge
right_merge = pd.merge(df_employees, df_salaries, on='EmployeeID', how='right')
print("\nRight merge:")
print(right_merge)

# Outer merge
outer_merge = pd.merge(df_employees, df_salaries, on='EmployeeID', how='outer')
print("\nOuter merge:")
print(outer_merge)

Inner merge:
   EmployeeID     Name Department  Salary
0           1    Alice      Sales   50000
1           2      Bob         IT   60000
2           3  Charlie         HR   55000

Left merge:
   EmployeeID     Name Department   Salary
0           1    Alice      Sales  50000.0
1           2      Bob         IT  60000.0
2           3  Charlie         HR  55000.0
3           4    David      Sales      NaN

Right merge:
   EmployeeID     Name Department  Salary
0           1    Alice      Sales   50000
1           2      Bob         IT   60000
2           3  Charlie         HR   55000
3           5      NaN        NaN   52000

Outer merge:
   EmployeeID     Name Department   Salary
0           1    Alice      Sales  50000.0
1           2      Bob         IT  60000.0
2           3  Charlie         HR  55000.0
3           4    David      Sales      NaN
4           5      NaN        NaN  52000.0


## 4. Pivot Tables

In [8]:
# Create pivot table
pivot = df_sales.pivot_table(
    values='Amount',
    index='Department',
    columns='Product',
    aggfunc='sum',
    fill_value=0
)

print("Pivot table (Amount by Department and Product):")
print(pivot)

# Pivot with multiple aggregations
pivot_multi = df_sales.pivot_table(
    values=['Amount', 'Quantity'],
    index='Department',
    columns='Product',
    aggfunc={'Amount': 'sum', 'Quantity': 'mean'}
)

print("\nPivot with multiple aggregations:")
print(pivot_multi)

Pivot table (Amount by Department and Product):
Product       A    B    C
Department               
HR            0    0  630
IT            0  845    0
Sales       460    0    0

Pivot with multiple aggregations:
           Amount               Quantity            
Product         A      B      C        A     B     C
Department                                          
HR            NaN    NaN  630.0      NaN   NaN  15.5
IT            NaN  845.0    NaN      NaN  21.0   NaN
Sales       460.0    NaN    NaN     11.5   NaN   NaN


## 5. Reshaping: Melt, Stack, Unstack

In [9]:
# Melt - wide to long format
df_wide = pd.DataFrame({
    'Date': ['2023-01', '2023-02'],
    'ProductA': [100, 120],
    'ProductB': [200, 210],
    'ProductC': [150, 160]
})

print("Wide format:")
print(df_wide)

df_long = df_wide.melt(
    id_vars=['Date'],
    value_vars=['ProductA', 'ProductB', 'ProductC'],
    var_name='Product',
    value_name='Amount'
)

print("\nLong format (after melt):")
print(df_long)

Wide format:
      Date  ProductA  ProductB  ProductC
0  2023-01       100       200       150
1  2023-02       120       210       160

Long format (after melt):
      Date   Product  Amount
0  2023-01  ProductA     100
1  2023-02  ProductA     120
2  2023-01  ProductB     200
3  2023-02  ProductB     210
4  2023-01  ProductC     150
5  2023-02  ProductC     160


In [10]:
# Stack - convert columns to rows
stacked = pivot.stack()
print("Stacked pivot table:")
print(stacked)

# Unstack - convert rows to columns
unstacked = stacked.unstack()
print("\nUnstacked (back to original):")
print(unstacked)

Stacked pivot table:
Department  Product
HR          A            0
            B            0
            C          630
IT          A            0
            B          845
            C            0
Sales       A          460
            B            0
            C            0
dtype: int64

Unstacked (back to original):
Product       A    B    C
Department               
HR            0    0  630
IT            0  845    0
Sales       460    0    0


## 6. Window Functions: Rolling and Expanding

In [11]:
# Create time series data
df_ts = df_sales.sort_values('Date').reset_index(drop=True)

# Rolling average
df_ts['rolling_avg_3'] = df_ts['Amount'].rolling(window=3).mean()
df_ts['rolling_sum_3'] = df_ts['Amount'].rolling(window=3).sum()

print("Rolling window operations:")
print(df_ts[['Date', 'Amount', 'rolling_avg_3', 'rolling_sum_3']].head(8))

Rolling window operations:
        Date  Amount  rolling_avg_3  rolling_sum_3
0 2023-01-01     100            NaN            NaN
1 2023-02-01     200            NaN            NaN
2 2023-03-01     150     150.000000          450.0
3 2023-04-01     120     156.666667          470.0
4 2023-05-01     210     160.000000          480.0
5 2023-06-01     160     163.333333          490.0
6 2023-07-01     110     160.000000          480.0
7 2023-08-01     220     163.333333          490.0


In [12]:
# Expanding window
df_ts['expanding_sum'] = df_ts['Amount'].expanding().sum()
df_ts['expanding_mean'] = df_ts['Amount'].expanding().mean()

print("Expanding window operations:")
print(df_ts[['Date', 'Amount', 'expanding_sum', 'expanding_mean']].head(8))

Expanding window operations:
        Date  Amount  expanding_sum  expanding_mean
0 2023-01-01     100          100.0      100.000000
1 2023-02-01     200          300.0      150.000000
2 2023-03-01     150          450.0      150.000000
3 2023-04-01     120          570.0      142.500000
4 2023-05-01     210          780.0      156.000000
5 2023-06-01     160          940.0      156.666667
6 2023-07-01     110         1050.0      150.000000
7 2023-08-01     220         1270.0      158.750000


## 7. Performance Tips

In [13]:
# Using categorical dtype for memory efficiency
print("Before categorical:")
print(f"Memory usage: {df_sales.memory_usage(deep=True).sum() / 1024:.2f} KB")

# Convert to categorical
df_cat = df_sales.copy()
df_cat['Department'] = df_cat['Department'].astype('category')
df_cat['Product'] = df_cat['Product'].astype('category')

print("\nAfter categorical:")
print(f"Memory usage: {df_cat.memory_usage(deep=True).sum() / 1024:.2f} KB")
print(f"Memory saved: {(1 - df_cat.memory_usage(deep=True).sum() / df_sales.memory_usage(deep=True).sum()) * 100:.1f}%")

Before categorical:
Memory usage: 1.79 KB

After categorical:
Memory usage: 0.92 KB
Memory saved: 48.7%


In [14]:
# Using eval for speed
import time

# Create larger dataset
large_df = pd.DataFrame({
    'A': np.random.rand(100000),
    'B': np.random.rand(100000),
    'C': np.random.rand(100000)
})

# Standard way
start = time.time()
result1 = large_df['A'] + large_df['B'] * large_df['C']
time1 = time.time() - start

# Using eval
start = time.time()
result2 = large_df.eval('A + B * C')
time2 = time.time() - start

print(f"Standard: {time1*1000:.3f} ms")
print(f"Using eval: {time2*1000:.3f} ms")
print(f"Speedup: {time1/time2:.1f}x")

Standard: 3.225 ms
Using eval: 16.106 ms
Speedup: 0.2x
